# NYC Public Level 2 EV Charging — Milestone 4 Gurobi Notebook

**IEOR 4004 · Team 34 · Current verified pipeline (November 4, 2026)**

This notebook documents and reports the **current Census-tract model**. It deliberately uses the authoritative implementation in `src/data_preparation.py` and `src/optimization.py` rather than maintaining a second, conflicting optimization model inside the notebook.

- **Base case:** 0.5-mile Haversine service radius, $3.75 million budget, $\lambda=1$, five years of 5% annual demand growth.
- **Municipal installation limit:** $U_j = \min(20,\lfloor 0.20\max(0,\text{parking spaces}_j-n_j^0)\rfloor)$ for all 40 eligible municipal sites.
- **Municipal charging capacity:** facility-specific operating hours $H_j$; other existing public Level 2 sites retain an *assumed* 168 hours/week.
- **Last checked-in tract result:** 14 facilities, **55 new ports**, **$601,935** modeled investment, **490,980.68 kWh/week** served, **49.97%** of projected demand. These are reference values from the checked-in solution, not hard-coded solver outputs.

**Default behavior:** read checked-in results without running Gurobi. Set `RUN_SOLVER = True` in Section 5 to run the current pipeline; the fresh run writes to `results/notebook_run/` so it does **not** overwrite the checked-in reference results. Open this notebook inside a local clone of the repository.

## 1. Setup and repository location

Run the notebook from the repository root (or from one of its subfolders). The notebook locates `run_model.py` automatically. Python dependencies come from `requirements.txt`; a fresh optimization run additionally requires a usable Gurobi license.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 160)

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "run_model.py").is_file() and (p / "config" / "tract_case.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook from a local clone of IEOR4004-project-Energy-Team34 "
        "(repository root or a subfolder)."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "config" / "tract_case.json"
REFERENCE_DIR = PROJECT_ROOT / "results" / "tract_case"
COMPARISON_CSV = PROJECT_ROOT / "results" / "geography_comparison" / "comparison.csv"
print("Repository:", PROJECT_ROOT)
print("Reference results:", REFERENCE_DIR)

## 2. Scenario parameters and capacity assumptions

Read parameters from the same config file used by the production script. `weekly_operating_hours = 168` in the config applies to **other public sites**, **not** universally to municipal facilities. Municipal operating hours come from the `Scheduled hours/week` field in `data/NYC_Municipal_Parking_Locations.xlsx` and are validated by the data-preparation module.

For municipal facility $j$:

$$q_j = 7.2\,H_j\times 0.90\quad\text{(kWh per added port per week)}$$

$$C_j^0=n_j^0 q_j, \qquad U_j=\min\left(20,\left\lfloor 0.20\max(0,P_j-n_j^0)\right\rfloor\right).$$

For other existing public Level 2 sites, the assumed existing-port capacity is $7.2\times168\times0.9=1088.64$ kWh/week.

In [ ]:
with CONFIG_PATH.open(encoding="utf-8") as f:
    config = json.load(f)

params = {
    "Budget (USD)": config["budget_usd"],
    "Service radius (miles)": config["service_radius_miles"],
    "Equity lambda": config["equity_lambda"],
    "Energy per EV (kWh/week)": config["weekly_energy_per_ev_kwh"],
    "Public L2 share": config["public_l2_share"],
    "Annual growth": config["annual_demand_growth_rate"],
    "Horizon (years)": config["planning_horizon_years"],
    "Port power (kW)": config["port_power_kw"],
    "Availability factor": config["availability_factor"],
    "Other public site weekly hours (assumed)": config["weekly_operating_hours"],
    "Maximum site installation share": config["max_new_port_share_of_remaining_spaces"],
    "Maximum additional ports per site": config["max_additional_ports_per_site"],
    "Fixed first-time cost (USD)": config["fixed_first_time_install_cost_usd"],
    "Additional port cost (USD)": config["incremental_port_cost_usd"],
}
assert config["demand_geography"] == "tract", "This notebook expects the tract config."
assert config["max_new_port_share_of_remaining_spaces"] == 0.2, "Review changed installation assumptions."
pd.DataFrame(params.items(), columns=["Parameter", "Current value"])

## 3. Demand, sites, accessible pairs, and equity inputs

The production pipeline allocates 2026 DMV ZIP-level electric-vehicle registration records to Census tracts with the HUD-USPS residential-address ratio (renormalized over matched NYC tracts). Each demand point is an ACS tract internal point; distances are point-to-point Haversine distances, not driving or walking distances.

For tract $i$, baseline demand is $D_i = \text{allocated EVs}_i\times 60\times0.25$ and projected demand is $D_i(1+r)^T$. Existing accessible supply is $S_i^0=\sum_{j:(i,j)\in A} C_j^0$ and the fixed shortage proxy is $g_i=\max(0,1-S_i^0/D_i)$ when $D_i>0$ (else 0). This is **charging-accessibility equity**, not demographic or income equity.

Below we use `prepare_model_data` rather than reimplementing geocoding, matching, and eligibility rules. Gun Hill is **included** under current data inputs; its announced closure remains an unresolved planning risk.

In [ ]:
from src.data_preparation import prepare_model_data

model_data = prepare_model_data(PROJECT_ROOT, CONFIG_PATH)
print(f"Demand points: {len(model_data.demand):,}")
print(f"Municipal investment candidates: {len(model_data.municipal):,}")
print(f"Total modeled supply sites: {len(model_data.sites):,}")
print(f"Accessible pairs: {len(model_data.accessible_pairs):,}")

municipal = model_data.municipal
print("Municipal operating hours range:",
      municipal["operating_hours_per_week"].min(), "to",
      municipal["operating_hours_per_week"].max(), "hours/week")
print("Municipal sites with fewer than 168 weekly hours:",
      int((municipal["operating_hours_per_week"] < 168).sum()))

columns = ["facility_id", "operating_hours_per_week", "existing_ports",
           "max_additional_ports", "added_port_capacity_kwh"]
municipal[columns].head(10)

## 4. Mathematical formulation (implemented in `src/optimization.py`)

**Sets:** $I$ = tract demand points; $M$ = municipal investment sites; $O$ = other existing supply sites; $A$ = accessible tract-site pairs.

**Decision variables:** $y_j\in\{0,1\}$ activates municipal investment; $x_j\in\mathbb Z_{\ge0}$ counts added ports; $f_{ij}\ge0$ represents weekly energy served along accessible pair $(i,j)$.

**Stage 1 objective:**
$$\max\quad Z=\sum_{(i,j)\in A}(1+\lambda g_i)f_{ij}.$$

**Constraints:**

1. **Budget:** fixed first-time installation costs plus port costs cannot exceed $B$.
2. **Activation and installation:** $y_j\le x_j\le U_j y_j$ for each municipal site.
3. **Projected demand:** $\sum_jf_{ij}\le D_i(1+r)^T$ for each tract.
4. **Municipal supply:** $\sum_if_{ij}\le C_j^0+q_j x_j$ for each municipal site.
5. **Other-site supply:** $\sum_if_{ij}\le C_j^0$ for each other public station.
6. **Accessibility and domains:** only pairs within the 0.5-mile radius have flow variables; variables respect their defined types and bounds.

**Stage 2:** Keep Stage 1's objective within the production code's small preservation tolerance and minimize investment cost. This is a *tie-break*, not a different primary objective. It need not reproduce an identical site allocation on every solver/environment if multiple near-optimal plans exist.

## 5. Optional live Gurobi solve

With `RUN_SOLVER = False`, no solver is launched: the rest of the notebook reads the committed tract outputs. To rerun the authoritative implementation, set it to `True`; output goes to `results/notebook_run` to avoid silently modifying checked-in reference files. A live solve requires `gurobipy` and an appropriate Gurobi license.

In [ ]:
RUN_SOLVER = False  # Set True only when you intend to rerun Gurobi.

if RUN_SOLVER:
    report_dir = PROJECT_ROOT / "results" / "notebook_run"
    command = [
        sys.executable, str(PROJECT_ROOT / "run_model.py"),
        "--config", str(CONFIG_PATH),
        "--output-dir", str(report_dir),
    ]
    print("Running:", " ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    report_dir = REFERENCE_DIR
    print("Reading checked-in results (no solver run):", report_dir)

assert (report_dir / "summary.json").is_file(), "Missing result summary.json"
assert (report_dir / "sanity_checks.json").is_file(), "Missing sanity_checks.json"

## 6. Tract solution and solver status

Read the summary returned by the actual model. Do **not** type expected counts or costs into notebook outputs: values below come directly from the selected output directory.

In [ ]:
with (report_dir / "summary.json").open(encoding="utf-8") as f:
    summary = json.load(f)
with (report_dir / "sanity_checks.json").open(encoding="utf-8") as f:
    sanity = json.load(f)

metrics = {
    "Demand points": summary["demand_points"],
    "Municipal candidates": summary["data_audit"]["municipal_modeled_candidates"],
    "Supply sites": summary["supply_sites"],
    "Accessible pairs": summary["accessible_pairs"],
    "Selected municipal facilities": summary["selected_municipal_sites"],
    "Added Level 2 ports": summary["additional_ports"],
    "Investment cost (USD)": summary["investment_cost_usd"],
    "Served demand (kWh/week)": summary["total_served_kwh"],
    "Projected demand (kWh/week)": summary["projected_demand_kwh"],
    "Service rate": summary["total_served_kwh"] / summary["projected_demand_kwh"],
    "Stage 1 weighted objective": summary["primary_stage1_objective"],
    "Unused budget (USD)": summary["budget_slack_usd"],
    "Stage 1 status": summary["primary_solver_status"],
    "Stage 2 status": summary["solver_status"],
    "Stage 1 MIP gap": summary["primary_mip_gap"],
    "Stage 2 MIP gap": summary["final_mip_gap"],
}
pd.DataFrame(metrics.items(), columns=["Metric", "Value"])

In [ ]:
print("Feasibility checks:")
for name, value in sanity.items():
    print(f"  {name}: {value}")

assert sanity["all_checks_pass"], "The saved solution failed at least one feasibility check."
assert sanity["maximum_flow_pair_distance_miles"] <= config["service_radius_miles"] + 1e-6
print("PASS: reported solution satisfies the automated checks.")

# These specific numbers characterize the checked-in 2026-11-04 reference run only.
if not RUN_SOLVER:
    assert summary["demand_points"] == 2285
    assert summary["data_audit"]["municipal_modeled_candidates"] == 40
    assert summary["additional_ports"] == 55
    assert summary["investment_cost_usd"] == 601935
    print("PASS: checked-in reference outputs match the verified November 4 base case.")

## 7. Selected municipal facilities

This table uses the actual exported solution. It shows investment choices; the candidate site count also includes sites where no new port was installed.

In [ ]:
plan = pd.read_csv(report_dir / "municipal_candidate_solution.csv")
selected = plan.loc[plan["additional_ports"].gt(0)].copy()
show = [c for c in [
    "facility_id", "site_id", "additional_ports", "investment_cost_usd",
    "served_kwh", "operating_hours_per_week", "max_additional_ports"
] if c in selected.columns]
print("Selected facilities:", len(selected))
print("Total additional ports:", int(selected["additional_ports"].sum()))
selected[show].sort_values("additional_ports", ascending=False)

## 8. ZIP versus Census-tract comparison

The ZIP case is a **comparison baseline**, not the principal Milestone 4 result. Equity shortage scores are recalculated for each geography, so equity-weighted objectives are not directly comparable across ZIP and tract cases.

In [ ]:
comparison = pd.read_csv(COMPARISON_CSV)
comparison["service_rate"] = (
    comparison["total_served_kwh"] / comparison["projected_demand_kwh"]
)
cols = [
    "demand_geography", "demand_points", "accessible_pairs",
    "selected_municipal_sites", "additional_ports", "investment_cost_usd",
    "total_served_kwh", "projected_demand_kwh", "service_rate", "all_checks_pass"
]
comparison[cols]

## 9. ZIP sensitivity scenarios (saved results)

These are the **ZIP-level** experiments produced by `run_sensitivity.py`; they do not replace the tract base case. The one-mile scenario should not be described as the adopted 0.5-mile base case. Its saved Stage 2 MIP gap is approximately 0.393%, not exact 0%.

In [ ]:
sensitivity_csv = PROJECT_ROOT / "results" / "sensitivity" / "scenario_comparison.csv"
if sensitivity_csv.is_file():
    sens = pd.read_csv(sensitivity_csv)
    columns = [c for c in [
        "scenario_name", "selected_municipal_sites", "additional_ports",
        "investment_cost_usd", "total_served_kwh", "projected_demand_kwh",
        "final_mip_gap", "all_sanity_checks_pass"
    ] if c in sens.columns]
    print(sens[columns].to_string(index=False))
else:
    print("No saved ZIP sensitivity comparison found. Run run_sensitivity.py if needed.")

## 10. Optional local Census-tract equity experiments

If you separately ran `run_model.py` for $\lambda\in\{0,1,2,4,10\}$ and saved the results in `results/tract_equity/lambdaN/`, this cell summarizes **local** experiments. The main repository does not necessarily contain those folders. It is not evidence of a run unless those output files exist.

In [ ]:
equity_rows = []
for lam in (0, 1, 2, 4, 10):
    path = PROJECT_ROOT / "results" / "tract_equity" / f"lambda{lam}" / "summary.json"
    if path.is_file():
        with path.open(encoding="utf-8") as f:
            s = json.load(f)
        equity_rows.append({
            "lambda": lam,
            "selected_sites": s["selected_municipal_sites"],
            "additional_ports": s["additional_ports"],
            "investment_cost_usd": s["investment_cost_usd"],
            "served_kwh_week": s["total_served_kwh"],
            "weighted_objective": s["primary_objective_equity_weighted_kwh"],
            "stage2_mip_gap": s["final_mip_gap"],
        })
if equity_rows:
    print(pd.DataFrame(equity_rows).to_string(index=False))
else:
    print("No local tract equity sensitivity outputs are available in results/tract_equity/.")

## 11. Interpretation, limitations, and reproducibility

- The solver's **OPTIMAL** status is a statement about this mathematical optimization model and its supplied data, **not** a promise of actual construction feasibility.
- The selected **55 added ports** are model decisions, not installed real-world chargers. Existing ports are separate from added ports.
- **Facility-specific municipal schedules** are inputs; scheduled parking-garage hours are not necessarily verified charger-access hours. Some sites require review, including Gun Hill's closure notice.
- Accessibility uses a **0.5-mile Haversine distance threshold** without separate screening for bridges, highways, waterways, or pedestrian routes.
- EV registrations are distributed from ZIP to tracts using HUD-USPS ratios. ACS tract internal points are not individual charging-demand addresses.
- The shortage score $g_i$ is a charging-accessibility proxy; it is not a measure of socioeconomic disadvantage.
- Costs are planning benchmarks, not current NYC contractor quotes. The $3.75 million budget is a scenario assumption, not a spending requirement.

**Reproduction commands (from repository root):**

```bash
python -m pip install -r requirements.txt
python run_model.py --config config/tract_case.json --output-dir results/tract_case
python run_geography_comparison.py
python run_sensitivity.py  # ZIP-level scenarios
python -m unittest discover -s tests
```

For authoritative interpretation, see `README.md`, `docs/MILESTONE4.md`, `MILESTONE4_REPORT_REMINDERS.md`, and `AGENTS.md`. The algebraic formulation should remain consistent with `docs/IEOR4004_Milestone3_Latest.pdf`.